# Music4All-Onion 数据集快速走查

这个 notebook 用于快速回答：`music4allOnion` 目录下每个数据文件是什么、长什么样。

会做三件事：
1. 列出所有数据文件
2. 读取 README 的官方描述
3. 对每个 `.tsv.bz2` 抽样查看列数和前两行样例

In [1]:
from pathlib import Path
import pandas as pd

# 自动定位数据目录
CANDIDATES = [
    Path.cwd() / "music4allOnion",
    Path("/Users/itsnotjerryh/Desktop/Github/Gen4Rec/music4allOnion"),
]

DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("找不到 music4allOnion 目录，请检查路径")

print("DATA_DIR:", DATA_DIR)

all_files = sorted([p.name for p in DATA_DIR.iterdir() if p.is_file()])

def infer_group(name: str) -> str:
    if name.startswith("userid_"):
        return "Listening events"
    if "lyrics" in name or "vad_" in name:
        return "EMD (lyrics/emotion)"
    if "tags" in name or "genres" in name:
        return "UGC/EGC (tags/genres)"
    if name.startswith("id_incp") or name.startswith("id_resnet") or name.startswith("id_vgg"):
        return "DC (video frame embeddings)"
    if name.endswith(".tsv.bz2"):
        return "Audio/Embedding features"
    if name.endswith(".tar.gz"):
        return "Archive"
    return "Other"

overview = pd.DataFrame({
    "file": all_files,
    "group": [infer_group(f) for f in all_files]
})

overview

DATA_DIR: /Users/itsnotjerryh/Desktop/Github/Gen4Rec/music4allOnion


,file,group
0,README.md,Other
1,id_bert.tsv.bz2,Audio/Embedding features
2,id_blf_correlation.tsv.bz2,Audio/Embedding features
3,id_blf_deltaspectral.tsv.bz2,Audio/Embedding features
4,id_blf_logfluc.tsv.bz2,Audio/Embedding features
5,id_blf_spectral.tsv.bz2,Audio/Embedding features
6,id_blf_spectralcontrast.tsv.bz2,Audio/Embedding features
7,id_blf_vardeltaspectral.tsv.bz2,Audio/Embedding features
8,id_chroma_bow.tsv.bz2,Audio/Embedding features
9,id_compare_audspec_stats.tsv.bz2,Audio/Embedding features


In [2]:
import bz2
import csv
import re

readme_path = DATA_DIR / "README.md"
readme_text = readme_path.read_text(encoding="utf-8", errors="ignore") if readme_path.exists() else ""

# 从 README 的 markdown 表格中提取 file -> description
desc_map = {}
for line in readme_text.splitlines():
    # 示例: | `id_blf_correlation.tsv.bz2` | [Correlation Pattern BLF](...) | Audio | ... |
    m = re.match(r"\|\s*`([^`]+)`\s*\|\s*(.+?)\s*\|\s*[^|]+\|", line)
    if m:
        fname = m.group(1).strip()
        desc = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", m.group(2)).strip()  # 去掉 markdown 超链接
        desc_map[fname] = desc

def sample_tsv_bz2(path: Path):
    with bz2.open(path, "rt", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f, delimiter="\t")
        row1 = next(reader, None)
        row2 = next(reader, None)
    row1 = row1 or []
    row2 = row2 or []
    return {
        "n_cols_row1": len(row1),
        "n_cols_row2": len(row2),
        "sample_row1": row1[:6],
        "sample_row2": row2[:6],
    }

records = []
for f in sorted(DATA_DIR.glob("*.tsv.bz2")):
    info = sample_tsv_bz2(f)
    records.append({
        "file": f.name,
        "group": infer_group(f.name),
        "description": desc_map.get(f.name, "(README 未匹配到描述，可能是命名差异)"),
        "n_cols_row1": info["n_cols_row1"],
        "n_cols_row2": info["n_cols_row2"],
        "sample_row1": " | ".join(info["sample_row1"]),
        "sample_row2": " | ".join(info["sample_row2"]),
    })

profile_df = pd.DataFrame(records)

# 展示：每个数据集长什么样
profile_df

,file,group,description,n_cols_row1,n_cols_row2,sample_row1,sample_row2
0,id_bert.tsv.bz2,Audio/Embedding features,(README 未匹配到描述，可能是命名差异),769,769,id | 0 | 1 | 2 | 3 | 4,9jbSytob9XRzwvB6 | 0.009224704466760159 | 0.04...
1,id_blf_correlation.tsv.bz2,Audio/Embedding features,Correlation Pattern BLF,1327,1327,id | BLF_CORR0000 | BLF_CORR0001 | BLF_CORR000...,0009fFIM1eYThaPg | 0.000320772 | 0.000137197 |...
2,id_blf_deltaspectral.tsv.bz2,Audio/Embedding features,Delta Spectral Pattern BLF,1373,1373,id | BLF_DELTASPEC0000 | BLF_DELTASPEC0001 | B...,0009fFIM1eYThaPg | 0.00139257 | 0.00196831 | 0...
3,id_blf_logfluc.tsv.bz2,Audio/Embedding features,Logarithmic Fluctuation Pattern BLF,3628,3628,id | ID | BLF_LOGFLUC0000 | BLF_LOGFLUC0001 | ...,0 | 0009fFIM1eYThaPg | 28.81 | 39.6568 | 40.01...
4,id_blf_spectral.tsv.bz2,Audio/Embedding features,Spectral Pattern BLF,981,981,id | BLF_SPEC0000 | BLF_SPEC0001 | BLF_SPEC000...,0009fFIM1eYThaPg | -0.0214604 | -0.0172305 | -...
5,id_blf_spectralcontrast.tsv.bz2,Audio/Embedding features,Spectral Contrast Pattern BLF,801,801,id | BLF_SPEC_CTRS000 | BLF_SPEC_CTRS001 | BLF...,0009fFIM1eYThaPg | 0.0190067 | 0.0232173 | 0.0...
6,id_blf_vardeltaspectral.tsv.bz2,Audio/Embedding features,Variance Delta Spectral Pattern BLF,1345,1345,id | BLF_VARDELTASPEC0000 | BLF_VARDELTASPEC00...,0009fFIM1eYThaPg | 0.0048828 | 0.00482754 | 0....
7,id_chroma_bow.tsv.bz2,Audio/Embedding features,BoAW of pitch-related features extracted with ...,501,501,id | cB000 | cB001 | cB002 | cB003 | cB004,ec73R8NdbZIZkrfL | 1.447158 | 1.5682018 | 0.0 ...
8,id_compare_audspec_stats.tsv.bz2,Audio/Embedding features,Statistical aggregation of ComParE hand-crafte...,2801,2801,id | audspec_lengthL1norm_sma_range | audspec_...,0009fFIM1eYThaPg | 6.960522 | 0.44336787 | 0.2...
9,id_compare_f0_stats.tsv.bz2,Audio/Embedding features,Statistical aggregation of ComParE hand-crafte...,84,84,id | F0final_sma_amean | F0final_sma_flatness ...,0009fFIM1eYThaPg | 176.70135 | 0.70742357 | 17...
